# Coding Agent

You'll need to build a Coding Agent powered by an LLM that can:
- Clone and explore GitHub repositories
- Read, analyze, and modify code files
- Execute tasks autonomously based on natural language instructions

In [ ]:
!pip install openai -q

#### Create OpenAI Client

In [ ]:
import os
from openai import OpenAI
from google.colab import userdata

try:
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    api_key = input("Enter your API key: ")

client = OpenAI(
    api_key=api_key
)

MODEL = "gpt-5-nano"

#### Clone Repo

In [ ]:
REPO_URL = "https://github.com/openai/openai-cookbook"

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

WORKSPACE = Path("/content/workspace")
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
REPO_PATH = WORKSPACE / REPO_NAME

# Prepare workspace
WORKSPACE.mkdir(parents=True, exist_ok=True)

# Remove existing copy if present (clean slate)
if REPO_PATH.exists():
    print(f"⚠️  Removing existing repo at {REPO_PATH}")
    shutil.rmtree(REPO_PATH)

# Clone (shallow clone for speed — depth=1 only fetches latest commit)
print(f"Cloning {REPO_URL} ...")
result = subprocess.run(
    ["git", "clone", "--depth", "1", REPO_URL, str(REPO_PATH)],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print("❌ Clone failed:")
    print(result.stderr)
else:
    print(f"✅ Repo cloned to: {REPO_PATH}")
    # Show top-level structure
    print("\n📂 Repository contents:")
    for item in sorted(REPO_PATH.iterdir()):
        marker = "📁" if item.is_dir() else "📄"
        print(f"  {marker} {item.name}")

# Change into the repo directory for subsequent operations
os.chdir(REPO_PATH)
print(f"\n Working directory: {os.getcwd()}")

Cloning https://github.com/openai/openai-cookbook ...
✅ Repo cloned to: /content/workspace/openai-cookbook

📂 Repository contents:
  📁 .git
  📁 .github
  📄 .gitignore
  📄 AGENTS.md
  📄 CONTRIBUTING.md
  📄 LICENSE
  📄 README.md
  📁 articles
  📄 authors.yaml
  📁 examples
  📁 images
  📄 registry.yaml

 Working directory: /content/workspace/openai-cookbook


## Instalar tavily

In [ ]:
!pip install tavily-python -q

## Configuración segura del workspace


In [ ]:
import json
import subprocess
from pathlib import Path

try:
    from tavily import TavilyClient
except Exception:
    TavilyClient = None

# Usamos el repo clonado como workspace principal.
# Si REPO_PATH ya existe por las celdas anteriores, usamos eso.
try:
    WORKSPACE = Path(REPO_PATH).resolve()
except NameError:
    WORKSPACE = Path("/content/workspace").resolve()

print(f"✅ Agent workspace: {WORKSPACE}")

✅ Agent workspace: /content/workspace/openai-cookbook


## Tools completas


In [ ]:
PROHIBITED_COMMANDS = [
    "rm -rf",
    "git push",
    "shutdown",
    "reboot",
    "sudo",
    "mkfs",
]


def resolve_path(path: str) -> Path:
    """
    Resuelve un path relativo dentro del WORKSPACE.
    Bloquea accesos fuera del workspace.
    """
    target = (WORKSPACE / path).resolve()

    if not str(target).startswith(str(WORKSPACE)):
        raise ValueError(f"Blocked path outside workspace: {path}")

    return target


def read_file(path: str) -> str:
    """Reads the content of a file given its path."""
    try:
        target = resolve_path(path)

        if not target.exists():
            return f"Error: File not found at {path}"

        if not target.is_file():
            return f"Error: {path} is not a file"

        content = target.read_text(encoding="utf-8")
        return content

    except Exception as e:
        return f"Error reading file {path}: {e}"


def write_file(path: str, content: str) -> str:
    """Writes content to a file, replacing its current content."""
    try:
        target = resolve_path(path)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")

        return f"Successfully wrote file: {path}"

    except Exception as e:
        return f"Error writing file {path}: {e}"


def list_files(path: str = ".") -> str:
    """Lists files in a directory."""
    try:
        target = resolve_path(path)

        if not target.exists():
            return f"Error: Directory not found at {path}"

        if not target.is_dir():
            return f"Error: {path} is not a directory"

        items = []
        for item in sorted(target.iterdir()):
            marker = "/" if item.is_dir() else ""
            items.append(item.name + marker)

        if not items:
            return f"Directory {path} is empty."

        return "\n".join(items)

    except Exception as e:
        return f"Error listing files in {path}: {e}"


def run_command(command: str) -> str:
    """Runs a terminal command inside the workspace and returns stdout/stderr."""
    try:
        lowered = command.lower()

        for forbidden in PROHIBITED_COMMANDS:
            if forbidden in lowered:
                return f"Blocked dangerous command: {command}"

        result = subprocess.run(
            command,
            shell=True,
            cwd=str(WORKSPACE),
            capture_output=True,
            text=True,
            timeout=60,
        )

        output = f"Exit code: {result.returncode}\n"
        output += "\n--- STDOUT ---\n"
        output += result.stdout or ""
        output += "\n--- STDERR ---\n"
        output += result.stderr or ""

        return output

    except subprocess.TimeoutExpired:
        return f"Error: command timed out: {command}"

    except Exception as e:
        return f"Error running command {command}: {e}"


def web_search(query: str) -> str:
    """Searches the web and returns results."""
    try:
        if TavilyClient is None:
            return "Error: tavily-python is not installed."

        try:
            tavily_api_key = userdata.get("TAVILY_API_KEY")
        except Exception:
            tavily_api_key = None

        if not tavily_api_key:
            return "Error: Missing TAVILY_API_KEY in Colab secrets."

        tavily_client = TavilyClient(api_key=tavily_api_key)

        response = tavily_client.search(
            query=query,
            max_results=5,
            search_depth="basic",
            include_answer=True,
        )

        return json.dumps(response, indent=2, ensure_ascii=False)

    except Exception as e:
        return f"Error searching the web: {e}"

## Definición de tools para el modelo

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read the content of a file inside the workspace.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "Relative file path inside the workspace."
                    }
                },
                "required": ["path"]
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Write content to a file inside the workspace.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "Relative file path inside the workspace."
                    },
                    "content": {
                        "type": "string",
                        "description": "Full content to write into the file."
                    }
                },
                "required": ["path", "content"]
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_command",
            "description": "Run a terminal command inside the workspace.",
            "parameters": {
                "type": "object",
                "properties": {
                    "command": {
                        "type": "string",
                        "description": "Terminal command to execute."
                    }
                },
                "required": ["command"]
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List files in a directory inside the workspace.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "Relative directory path inside the workspace."
                    }
                },
                "required": ["path"]
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search information on the web.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query."
                    }
                },
                "required": ["query"]
            },
        },
    },
]

TOOL_FUNCTIONS = {
    "read_file": read_file,
    "write_file": write_file,
    "run_command": run_command,
    "list_files": list_files,
    "web_search": web_search,
}

MODIFYING_TOOLS = {"write_file", "run_command"}

## Harness del agente

In [ ]:
SYSTEM_PROMPT = """
You are a Coding Agent running in Google Colab.

You can solve coding tasks by using tools:
- read_file
- write_file
- list_files
- run_command
- web_search

Rules:
- Work only inside the workspace.
- Inspect files before modifying them when needed.
- Do not invent test results. If you need to know whether tests pass, run them.
- Use simple and safe terminal commands.
- When the task is complete, answer with a clear summary of what you did.
"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT}
]


def create_plan(user_message: str) -> str:
    """Creates a plan before executing tools."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "Create a short numbered plan to solve the task. Do not use tools."
            },
            {
                "role": "user",
                "content": user_message
            }
        ],
    )

    return response.choices[0].message.content


def approve_plan(user_message: str):
    """Shows a plan to the user and asks for approval."""
    while True:
        plan = create_plan(user_message)

        print("\n===== PLAN MODE =====")
        print(plan)
        print("=====================")

        decision = input("Approve this plan? Type 'yes', 'no', or 'modify': ").strip().lower()

        if decision in ["yes", "y", "si", "sí", "s"]:
            return plan

        if decision in ["no", "n"]:
            print("Plan rejected. Task cancelled.")
            return None

        if decision == "modify":
            modification = input("Write your modification: ")
            user_message = user_message + "\n\nUser modification: " + modification
            continue

        print("Invalid answer. Please type yes, no, or modify.")


def execute_tool_call(tool_call, supervision_mode: bool = True) -> str:
    """Executes a tool requested by the model."""
    tool_name = tool_call.function.name

    try:
        args = json.loads(tool_call.function.arguments or "{}")
    except Exception as e:
        return f"Error parsing tool arguments: {e}"

    if tool_name not in TOOL_FUNCTIONS:
        return f"Error: unknown tool {tool_name}"

    if supervision_mode and tool_name in MODIFYING_TOOLS:
        print("\n===== HUMAN SUPERVISION =====")
        print(f"The agent wants to execute: {tool_name}")
        print("Arguments:")
        print(json.dumps(args, indent=2, ensure_ascii=False))
        print("=============================")

        decision = input("Allow this action? Type 'yes' or 'no': ").strip().lower()

        if decision not in ["yes", "y", "si", "sí", "s"]:
            return f"Action {tool_name} rejected by user."

    result = TOOL_FUNCTIONS[tool_name](**args)
    return str(result)


def run_agent_turn(
    user_message: str,
    plan_mode: bool = True,
    supervision_mode: bool = True,
    max_iterations: int = 15,
):
    """
    Runs one agent turn.
    Keeps conversation history between turns.
    """
    if plan_mode:
        approved_plan = approve_plan(user_message)

        if approved_plan is None:
            return {
                "final_answer": "Task cancelled by user.",
                "model_iterations": 0,
                "tool_calls": 0,
            }

        user_message = f"""
User task:
{user_message}

Approved plan:
{approved_plan}
"""

    messages.append({"role": "user", "content": user_message})

    model_iterations = 0
    tool_calls_count = 0

    for _ in range(max_iterations):
        model_iterations += 1

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
            parallel_tool_calls=False,
        )

        assistant_message = response.choices[0].message
        messages.append(assistant_message.model_dump(exclude_none=True))

        if not assistant_message.tool_calls:
            final_answer = assistant_message.content or ""
            return {
                "final_answer": final_answer,
                "model_iterations": model_iterations,
                "tool_calls": tool_calls_count,
            }

        for tool_call in assistant_message.tool_calls:
            tool_calls_count += 1

            print(f"\n🔧 Tool requested: {tool_call.function.name}")

            tool_result = execute_tool_call(
                tool_call,
                supervision_mode=supervision_mode,
            )

            print("Tool result:")
            print(tool_result[:1500])

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": tool_result,
                }
            )

    return {
        "final_answer": "Max iterations reached without a final answer.",
        "model_iterations": model_iterations,
        "tool_calls": tool_calls_count,
    }

## Modo conversación

In [ ]:
def chat(plan_mode: bool = True, supervision_mode: bool = True):
    print("Coding Agent started.")
    print("Type 'exit' to stop.")
    print(f"Plan mode: {plan_mode}")
    print(f"Supervision mode: {supervision_mode}")

    while True:
        user_message = input("\nYou: ")

        if user_message.strip().lower() in ["exit", "quit", "salir"]:
            print("Chat finished.")
            break

        result = run_agent_turn(
            user_message=user_message,
            plan_mode=plan_mode,
            supervision_mode=supervision_mode,
        )

        print("\n===== FINAL ANSWER =====")
        print(result["final_answer"])
        print("========================")
        print(f"Model iterations: {result['model_iterations']}")
        print(f"Tool calls: {result['tool_calls']}")

## Arrancar el agente

In [ ]:
chat(plan_mode=True, supervision_mode=True)

Coding Agent started.
Type 'exit' to stop.
Plan mode: True
Supervision mode: True

You: Quien es el capitan de la seleccion argentina de futbol?

===== PLAN MODE =====
Plan corto:
1) Identificar qué pregunta se hace: el capitán de la selección argentina.
2) Dar la respuesta basada en la información disponible hasta 2024.
3) Añadir nota de posibles variaciones y ofrecer precisión por competición si se solicita.

Respuesta:
Lionel Messi. Hasta mi última actualización en 2024, Messi es el capitán de la selección argentina. En algunos partidos podría ser suplente o no estar, y en esos casos el brazalete suele llevárselo otro jugador como Nicolás Otamendi o Ángel Di María. Si me dices la competición o el año, te digo quién llevó el brazalete en ese partido concreto.
Approve this plan? Type 'yes', 'no', or 'modify': yes

===== FINAL ANSWER =====
Lionel Messi.

Hasta mi última actualización en 2024, Messi es el capitán de la selección argentina. En algunos partidos podría no estar o ser suple

KeyboardInterrupt: Interrupted by user

### Coding Agent Implementation

This section will implement the Coding Agent according to the requirements: a harness for LLM interaction, conversation mode, several tools, and optional plan/supervision modes.

Let's start by defining the `read_file` tool.

In [ ]:
def read_file(path: str) -> str:
    """Reads the content of a file given its path."""
    try:
        with open(path, 'r') as f:
            content = f.read()
        print(f"✅ Successfully read file: {path}")
        return content
    except FileNotFoundError:
        print(f"❌ Error: File not found at {path}")
        return f"Error: File not found at {path}"
    except Exception as e:
        print(f"❌ Error reading file {path}: {e}")
        return f"Error reading file {path}: {e}"

# Example usage (for testing purposes)
# file_content = read_file('README.md')
# print('\n--- File Content (first 200 chars) ---')
# print(file_content[:200])
# print('-------------------------------------')


# Informe — Coding Agent

## 1. Objetivo

El objetivo del trabajo fue implementar un **Coding Agent** en Google Colab.  
El agente funciona como un **harness**, es decir, un loop que conecta un modelo de lenguaje con herramientas locales para resolver tareas de programación de forma autónoma.

El agente recibe una instrucción en lenguaje natural, puede usar herramientas, devuelve resultados al modelo y repite el proceso hasta completar la tarea.

---

## 2. Implementación

Se trabajó sobre el notebook:

```text
Copia de coding_agent.ipynb
```

El agente fue configurado para usar un modelo de OpenAI y trabajar sobre un repositorio clonado en Google Colab.

Repositorio usado:

```text
https://github.com/openai/openai-cookbook
```

Las herramientas implementadas fueron:

- `read_file`: lee el contenido de un archivo.
- `write_file`: crea o modifica archivos.
- `run_command`: ejecuta comandos de terminal y devuelve `stdout` y `stderr`.
- `list_files`: lista archivos de un directorio.
- `web_search`: busca información en la web.

También se implementaron dos modos importantes:

- **Plan mode**: antes de ejecutar una tarea, el agente propone un plan y pide aprobación.
- **Supervision mode**: antes de escribir archivos o ejecutar comandos, el agente pide autorización humana.

---

## 3. Corrida 1: exploración del repositorio

### Tarea enviada

```text
Explore this repository and explain briefly what it contains. Use the available tools to inspect the files.
```

### Desarrollo

El agente propuso un plan para inspeccionar la estructura del repositorio, revisar archivos principales, documentación y configuraciones.  
El plan fue aprobado por el usuario.

Durante la ejecución, el agente utilizó principalmente:

```text
list_files
read_file
```

Inspeccionó archivos y carpetas como:

```text
.git/
.github/
.gitignore
AGENTS.md
CONTRIBUTING.md
LICENSE
README.md
articles/
authors.yaml
examples/
images/
registry.yaml
```

También leyó archivos como `README.md`, `registry.yaml`, `authors.yaml` y scripts dentro de `.github`.

### Resultado

La primera corrida no terminó con una respuesta final completa porque el agente alcanzó el límite máximo de iteraciones.

```text
Final answer: Max iterations reached without a final answer.
Model iterations: 15
Tool calls: 15
```

### Qué salió bien

El agente pudo listar archivos, leer documentación y navegar la estructura del repositorio.  
También logró identificar que el repositorio `openai-cookbook` contiene ejemplos, notebooks, artículos, imágenes, archivos de autores y configuraciones relacionadas con documentación y workflows.

### Qué salió mal

La tarea era muy amplia y el agente siguió explorando hasta alcanzar el máximo de iteraciones.  
Una mejora sería limitar la exploración a algunos archivos principales, como `README.md`, `AGENTS.md` y `registry.yaml`, y luego pedirle que responda.

---

## 4. Corrida 2: creación de código y tests

### Tarea enviada

```text
Create a file called simple_calculator.py with functions add, subtract, multiply and divide. Then create a pytest file called test_simple_calculator.py and run the tests. If something fails, fix it.
```

### Desarrollo

El agente propuso un plan para:

1. Crear `simple_calculator.py`.
2. Implementar las funciones `add`, `subtract`, `multiply` y `divide`.
3. Crear `test_simple_calculator.py` con tests usando `pytest`.
4. Ejecutar los tests.
5. Corregir errores si aparecían.

El plan fue aprobado por el usuario.

Como estaba activado el modo de supervisión, el agente pidió permiso antes de escribir archivos y antes de ejecutar comandos.

---

## 5. Archivos creados

El agente creó el archivo:

```text
simple_calculator.py
```

con funciones de suma, resta, multiplicación y división.

También creó:

```text
test_simple_calculator.py
```

con tests para validar:

- suma;
- resta;
- multiplicación;
- división;
- división por cero.

---

## 6. Ejecución de tests

Primero, el agente ejecutó:

```text
pytest -q
```

Ese comando falló porque intentó correr todos los tests del repositorio `openai-cookbook`.  
Los errores no estaban relacionados con el archivo creado, sino con tests y dependencias propias del repositorio.

Luego el agente corrigió el comando y ejecutó solo los tests creados:

```text
pytest -q test_simple_calculator.py -q
```

El resultado fue exitoso:

```text
....                                                                     [100%]
Exit code: 0
```

Esto significa que los 4 tests pasaron correctamente.

### Resultado de la segunda corrida

```text
Model iterations: 5
Tool calls: 4
```

---

## 7. Qué salió bien en la segunda corrida

El agente logró completar la tarea correctamente.  
Pudo crear código, escribir tests, ejecutar comandos y validar que los tests pasaran.

También funcionó correctamente el modo de supervisión humana, ya que pidió autorización antes de usar `write_file` y `run_command`.

---

## 8. Qué salió mal en la segunda corrida

El primer comando de test fue demasiado general:

```text
pytest -q
```

Ese comando intentó correr todos los tests del repositorio y generó errores externos a la tarea.

Después, el agente corrigió esto ejecutando solo el archivo de test correspondiente.

---

## 9. Comparación de corridas

| Corrida | Tarea | Iteraciones del modelo | Tool calls | Resultado |
|---|---|---|---|---|
| 1 | Explorar repositorio | 15 | 15 | No terminó; alcanzó el máximo de iteraciones |
| 2 | Crear calculadora y tests | 5 | 4 | Terminó correctamente; 4 tests pasaron |

---

## 10. Conclusión

El Coding Agent implementado cumple con la idea principal del trabajo.  
Puede recibir instrucciones en lenguaje natural, generar un plan, pedir aprobación, usar herramientas y resolver tareas de programación.

La primera corrida mostró que las tareas muy amplias pueden consumir muchas iteraciones sin llegar a una respuesta final.  
La segunda corrida fue exitosa porque la tarea era concreta y verificable.

El agente demostró las siguientes funcionalidades:

- uso de tools;
- modo conversación;
- plan mode;
- supervisión humana;
- lectura de archivos;
- creación de archivos;
- ejecución de comandos;
- validación con tests.